In [2]:
from qdrant_client import QdrantClient
from qdrant_client.http import models as qm
from sentence_transformers import SentenceTransformer
from datasets import load_dataset

model = SentenceTransformer("all-MiniLM-L6-v2")
qc = QdrantClient("http://localhost:6333")

qc.recreate_collection(
    collection_name="imdb",
    vectors_config=qm.VectorParams(size=384, distance=qm.Distance.COSINE),
)

df = load_dataset("stanfordnlp/imdb", split="train").shuffle(seed=42).select(range(5000)).to_pandas()
docs = df["text"].tolist()
sentiments = ["positive" if l == 1 else "negative" for l in df["label"]]

vectors = model.encode(docs, normalize_embeddings=True, show_progress_bar=True).tolist()

qc.upload_points(      # CHANGED: uploads in batches instead of one huge request
    collection_name="imdb",
    points=[qm.PointStruct(id=i, vector=v, payload={"text": d, "sentiment": s})
            for i, (d, v, s) in enumerate(zip(docs, vectors, sentiments))],
    batch_size=256,
)
print("Stored:", qc.count("imdb").count)

c:\Users\a4saa\ai-trainee\.venv\Lib\site-packages\sentence_transformers\cross_encoder\CrossEncoder.py:11: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from tqdm.autonotebook import tqdm, trange
c:\Users\a4saa\ai-trainee\.venv\Lib\site-packages\transformers\tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(
C:\Users\a4saa\AppData\Local\Temp\ipykernel_23808\425119497.py:9: DeprecationWarning: `recreate_collection` method is deprecated and will be removed in the future. Use `collection_exists` to check collection existence and `create_collection` instead.
  qc.recreate_collection(
Batches: 100%|██████████| 157/157 

Stored: 5000


In [2]:
q = model.encode(["a film with breathtaking cinematography"], normalize_embeddings=True)[0].tolist()
hits = qc.search(
    collection_name="imdb", query_vector=q, limit=5,
    query_filter=qm.Filter(must=[qm.FieldCondition(key="sentiment", match=qm.MatchValue(value="positive"))]),
)
for h in hits:
    print(round(h.score, 3), h.payload["text"][:120])

0.573 It's hard to say sometimes why exactly a film is so effective. From the moment I first came across "The Stone Boy", some
0.566 I first saw this at a foreign film festival. It's a beautifully paced nail-biter about a plot to relieve the Estonian tr
0.566 The beginning of this film is a little clunky and also confusing , but sit tight, because you are in for the ride of you
0.562 I had pleasure to watch the short film "The Cure", by first time director Ryan Jafri. What really impress me are the cam
0.562 This is the definite Lars von Trier Movie, my favorite, I rank it higher than "Breaking the waves" or the latest "Dancer


In [3]:
from elasticsearch import Elasticsearch, helpers

es = Elasticsearch("http://localhost:9200")
if es.indices.exists(index="imdb_hybrid"):
    es.indices.delete(index="imdb_hybrid")

es.indices.create(index="imdb_hybrid", mappings={
    "properties": {
        "text":      {"type": "text", "analyzer": "english"},
        "sentiment": {"type": "keyword"},
        "embedding": {"type": "dense_vector", "dims": 384, "index": True, "similarity": "cosine"},
    }
})

ObjectApiResponse({'acknowledged': True, 'shards_acknowledged': True, 'index': 'imdb_hybrid'})

In [4]:
def gen():
    for i, (text, sent, vec) in enumerate(zip(docs, sentiments, vectors)):
        yield {"_index": "imdb_hybrid", "_id": i,
               "_source": {"text": text, "sentiment": sent, "embedding": vec}}

helpers.bulk(es, gen(), chunk_size=500)
es.indices.refresh(index="imdb_hybrid")
print("Docs:", es.count(index="imdb_hybrid")["count"])

Docs: 5000


In [5]:
q_vec = model.encode(["a film with breathtaking cinematography"], normalize_embeddings=True)[0].tolist()
r = es.search(index="imdb_hybrid", knn={
    "field": "embedding", "query_vector": q_vec, "k": 10, "num_candidates": 100,
})
for h in r["hits"]["hits"][:5]:
    print(round(h["_score"], 3), h["_source"]["text"][:120])

0.787 It's hard to say sometimes why exactly a film is so effective. From the moment I first came across "The Stone Boy", some
0.784 I first saw this at a foreign film festival. It's a beautifully paced nail-biter about a plot to relieve the Estonian tr
0.783 The beginning of this film is a little clunky and also confusing , but sit tight, because you are in for the ride of you
0.783 I really wanted to like this movie because the critics have been unkind<br /><br />to it (to say the least)... but it wa
0.783 I had pleasure to watch the short film "The Cure", by first time director Ryan Jafri. What really impress me are the cam


In [11]:
r = es.search(index="imdb_hybrid",
    query={"match": {"text": "breathtaking cinematography"}},
    knn={"field": "embedding", "query_vector": q_vec, "k": 10, "num_candidates": 100},
    rank={"rrf": {"rank_window_size": 50, "rank_constant": 60}},   # CHANGED: window_size → rank_window_size
    size=10,
)
for h in r["hits"]["hits"][:5]:
    print(h["_rank"], h["_source"]["text"][:120])

1 This is one of the best movies I've ever seen. It has very good acting by Hanks, Newman, and everyone else. Definitely J
2 I really wanted to like this movie because the critics have been unkind<br /><br />to it (to say the least)... but it wa
3 The pioneering Technicolor Cinematography (Winner of Special Technical Achievement Oscar) is indeed enchanting. Add an e
4 It's hard to say sometimes why exactly a film is so effective. From the moment I first came across "The Stone Boy", some
5 Why didn't this pick up a bag full of Oscars? It is an amazing interpretaion of an oft-filmed/performed piece. The visua


In [7]:
es.license.post_start_trial(acknowledge=True)

ObjectApiResponse({'acknowledged': True, 'trial_was_started': True, 'type': 'trial'})

In [1]:
def rrf(result_lists, k=60, top_n=10):
    scores = {}
    for ranks in result_lists:
        for rank, doc_id in enumerate(ranks, start=1):
            scores[doc_id] = scores.get(doc_id, 0) + 1 / (k + rank)
    return sorted(scores.items(), key=lambda x: -x[1])[:top_n]

In [3]:
import random
random.seed(7)
short = [i for i, d in enumerate(docs) if 300 < len(d) < 900]
for i in random.sample(short, 25):
    print(f"ID {i}: {docs[i][:300]}\n")

ID 3187: Modern, original, romantic story.<br /><br />Very good acting of both Nicole Kidman and Ben Chaplin.<br /><br />Miss Kidman does a nice job in imitating a Russian accent. Ben Chaplin is also good as the shy, dull clerk. For the men (and some women) : miss Kidman looks fantastic and is very sympathet

ID 1474: This film reminds me of how college students used to protest against the Vietnam War. As if, upon hearing some kids were doing without cheeseburgers in Cow Dung Collehe, the President was going to immediately change all US foreign policy.<br /><br />The worst thing is that, while dangerous, the conc

ID 3883: Absolutely putrid slasher film has not one redeeming quality. It has Camp Councellor Angela(Pamela Anderson..awful as the killer;her quips when she wastes people aren't even amusing)on the warpath slaying teenagers who act the least bit naughty or resist her pleadings for good behavior. We run the g

ID 477: When I first saw a small scene of it in some announcements,